In [ ]:
!pip install -q torch datasets huggingface-hub safetensors transformers numpy

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import BertModel, BertTokenizer
import numpy as np
import math
import time
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass

import warnings

warnings.filterwarnings("ignore")
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")

Using device: mps


# 1) ColBERT: Contextualized Late Interaction over BERT

<img src='https://weaviate.io/assets/images/colbert-16b13c39b2d90e59949f45c61be24271.jpg' width=500>

ColBERT's key insight: **delay the interaction between query and document to the very last step** — a simple-but-expressive Maximum Similarity (MaxSim) operator. 
This preserves fine-grained token-level matching while still allowing offline document encoding.

ColBERT's architecture is deceptively simple.
Both queries and documents are encoded independently, but instead of pooling into a single vector (as in a bi-encoder), **every token's contextualized embedding is retained**.

The architecture has three components:

1. **Query Encoder** `f_Q`: Maps a query string to a matrix of token embeddings
2. **Document Encoder** `f_D`: Maps a document string to a matrix of token embeddings
3. **Late Interaction (MaxSim)**: Computes relevance from the two embedding matrices

### Why a fixed query length? (Query Augmentation)
Queries are padded to a fixed length with `[MASK]` tokens. The released ColBERTv2 does not
let other tokens attend *to* the masks, but every `[MASK]` position attends to the real query
tokens and its output embedding joins MaxSim, so the model learns **soft expansion
embeddings** — new "virtual" query terms that capture related concepts. This is a form of learned query expansion
that improves recall at no computational cost during retrieval.

### Why filter punctuation from documents?
Punctuation tokens carry little semantic content but would increase the index size (one embedding
per token). Filtering them reduces storage without meaningfully hurting effectiveness.

In [2]:
# === Configuration ===
@dataclass
class ColBERTConfig:

    # Released ColBERTv2 checkpoint: bert-base-uncased backbone + Linear(768 -> 128)
    checkpoint: str = "colbert-ir/colbertv2.0"
    # BERT hidden size (768 for bert-base)
    bert_hidden_dim: int = 768

    # ColBERT projection dimension — paper uses 128
    embedding_dim: int = 128

    # Fixed query length for query augmentation (checkpoint: 32)
    query_max_len: int = 32
    # Maximum document length (checkpoint: 180)
    doc_max_len: int = 180

    # Whether to mask punctuation embeddings in documents
    mask_punctuation: bool = True
    # Punctuation characters to filter
    punctuation_tokens: str = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"


config = ColBERTConfig()
print(
    f"Config: embedding_dim={config.embedding_dim}, "
    f"query_max_len={config.query_max_len}, "
    f"doc_max_len={config.doc_max_len}"
)

Config: embedding_dim=128, query_max_len=32, doc_max_len=180


In [3]:
class ColBERT(nn.Module):
    """
    ColBERT: Contextualized Late Interaction over BERT.

    Architecture:
      Shared BERT backbone → Linear projection (768 → 128) → L2 normalization
      Query path:  [CLS] [Q] tokens... [MASK]... (fixed length N_q)
      Doc path:    [CLS] [D] tokens...           (variable length, punct filtered)

    [Q] and [D] are BERT's unused vocab entries [unused0] and [unused1]. As in the reference
    implementation, the text is prefixed with ". " and that placeholder token (position 1)
    is overwritten with the marker, so no real token is lost.
    """

    def __init__(self, cfg: ColBERTConfig):
        super().__init__()
        self.cfg = cfg

        # Pre-trained backbone (the checkpoint's "bert.*" weights)
        self.bert = BertModel.from_pretrained(cfg.checkpoint, add_pooling_layer=False)
        self.tokenizer = BertTokenizer.from_pretrained(cfg.checkpoint)

        # Linear projection: compress BERT's hidden dim to ColBERT's embedding dim
        # (batch_num, seq_len, bert_hidden_dim) → (batch_num, seq_len, embedding_dim)
        self.linear = nn.Linear(cfg.bert_hidden_dim, cfg.embedding_dim, bias=False)
        weights = load_file(hf_hub_download(cfg.checkpoint, "model.safetensors"))
        self.linear.weight.data.copy_(weights["linear.weight"])

        # [unused0] / [unused1] serve as the [Q] / [D] markers (ids 1 and 2)
        self.query_marker_id = self.tokenizer.convert_tokens_to_ids("[unused0]")
        self.doc_marker_id = self.tokenizer.convert_tokens_to_ids("[unused1]")
        self.mask_token_id = self.tokenizer.mask_token_id

        # Pre-compute set of punctuation token IDs for document filtering
        self.punct_ids = set()
        if cfg.mask_punctuation:
            for char in cfg.punctuation_tokens:
                ids = self.tokenizer.encode(char, add_special_tokens=False)
                self.punct_ids.update(ids)

    def _encode_query_tokens(
        self, queries: List[str]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Tokenize queries with [Q] marker and pad to fixed length with [MASK].

        Returns:
            input_ids:      (batch_num, query_max_len)
            attention_mask: (batch_num, query_max_len)

        The fixed-length padding with [MASK] is the "query augmentation" trick
        (Khattab & Zaharia, 2020, §3.3).
        """
        encoding = self.tokenizer(
            [". " + q for q in queries],
            padding="max_length",
            truncation=True,
            max_length=self.cfg.query_max_len,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        # Position 0 stays [CLS]; position 1 (the "." placeholder) becomes [Q]
        input_ids[:, 1] = self.query_marker_id

        # Replace [PAD] with [MASK]. The attention mask stays 0 there (the checkpoint's
        # attend_to_mask_tokens=False): masks attend to the query but are not attended to,
        # and their output embeddings still take part in MaxSim
        input_ids[input_ids == self.tokenizer.pad_token_id] = self.mask_token_id
        return input_ids, attention_mask

    def _encode_doc_tokens(
        self, docs: List[str]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Tokenize documents with [D] marker. No fixed-length padding (variable length).

        Returns:
            input_ids:      (batch_num, padded_doc_len)
            attention_mask: (batch_num, padded_doc_len)
            punct_mask:     (batch_num, padded_doc_len) — 1 for non-punct, 0 for punct
        """
        encoding = self.tokenizer(
            [". " + d for d in docs],
            padding="longest",
            truncation=True,
            max_length=self.cfg.doc_max_len,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        # Position 1 (the "." placeholder) becomes [D]
        input_ids[:, 1] = self.doc_marker_id

        # Build punctuation mask: 1 for tokens to keep, 0 for punctuation to filter
        punct_mask = torch.ones_like(input_ids, dtype=torch.float)
        if self.cfg.mask_punctuation:
            for pid in self.punct_ids:
                punct_mask[input_ids == pid] = 0.0

        return input_ids, attention_mask, punct_mask

    def encode_query(self, queries: List[str]) -> torch.Tensor:
        """
        Full query encoding pipeline: tokenize → BERT → project → normalize.

        Args:
            queries: List of query strings, length batch_num

        Returns:
            Q: (batch_num, query_max_len, embedding_dim) — L2-normalized embeddings
        """
        input_ids, attention_mask = self._encode_query_tokens(queries)

        # (batch_num, query_max_len) → (batch_num, query_max_len, bert_hidden_dim)
        hidden = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

        # (batch_num, query_max_len, bert_hidden_dim) → (batch_num, query_max_len, embedding_dim)
        projected = self.linear(hidden)

        # L2 normalize each token embedding: dot product = cosine similarity
        return F.normalize(projected, p=2, dim=-1)

    def encode_document(self, docs: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Full document encoding pipeline: tokenize → BERT → project → normalize → mask.

        Args:
            docs: List of document strings, length batch_num

        Returns:
            D:    (batch_num, padded_doc_len, embedding_dim) — L2-normalized embeddings
            mask: (batch_num, padded_doc_len) — 1 for real non-punct tokens, 0 otherwise
        """
        input_ids, attention_mask, punct_mask = self._encode_doc_tokens(docs)

        # (batch_num, padded_doc_len) → (batch_num, padded_doc_len, bert_hidden_dim)
        hidden = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

        # (batch_num, padded_doc_len, embedding_dim), L2-normalized
        D = F.normalize(self.linear(hidden), p=2, dim=-1)

        # Both padding and punctuation are dropped from the index
        # (batch_num, padded_doc_len)
        combined_mask = attention_mask.float() * punct_mask

        return D, combined_mask

    def forward(self, queries: List[str], docs: List[str]) -> torch.Tensor:
        """
        All-pairs ColBERT scores between a batch of queries and a batch of documents.

        Returns:
            scores: (batch_num, num_docs) — relevance scores
        """
        Q = self.encode_query(queries)
        D, D_mask = self.encode_document(docs)
        return self.maxsim(Q, D, D_mask)

    @staticmethod
    def maxsim(Q: torch.Tensor, D: torch.Tensor, D_mask: torch.Tensor) -> torch.Tensor:
        """
        MaxSim: the late-interaction operator. For each query token, find the document
        token it matches best, then sum these best-match scores over query tokens.

        Args:
            Q:      (batch_num, query_max_len, embedding_dim)
            D:      (num_docs, doc_len, embedding_dim)
            D_mask: (num_docs, doc_len) — 1 for real tokens, 0 for padding/punct

        Returns:
            scores: (batch_num, num_docs)
        """
        # Token-level cosine similarities (both sides are L2-normalized)
        # (batch_num, num_docs, query_max_len, doc_len)
        sim = torch.einsum("bqe,dje->bdqj", Q, D)

        # Masked document tokens can never win the max
        # D_mask: (num_docs, doc_len) → (1, num_docs, 1, doc_len)
        sim = sim.masked_fill(D_mask[None, :, None, :] == 0, -9999.0)

        # (batch_num, num_docs, query_max_len) → (batch_num, num_docs)
        return sim.max(dim=-1).values.sum(dim=-1)

In [4]:
# === Test ===
model = ColBERT(config).to(DEVICE).eval()

# Quick shape and sanity check with the trained weights
test_queries = ["What is the capital of France?"]
test_docs = [
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany.",
]

with torch.no_grad():
    Q = model.encode_query(test_queries)
    D, D_mask = model.encode_document(test_docs)
    scores = ColBERT.maxsim(Q, D, D_mask)

print(f"Query embeddings:    {Q.shape}")  # (1, 32, 128)
print(f"Doc embeddings:      {D.shape}")  # (2, doc_len, 128)
print(f"Doc mask:            {D_mask.shape}")  # (2, doc_len)
print(f"Scores:              {scores.shape}")  # (1, 2)
print(f"Score for Doc A (Paris):  {scores[0, 0].item():.4f}")
print(f"Score for Doc B (Berlin): {scores[0, 1].item():.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 
linear.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query embeddings:    torch.Size([1, 32, 128])
Doc embeddings:      torch.Size([2, 14, 128])
Doc mask:            torch.Size([2, 14])
Scores:              torch.Size([1, 2])
Score for Doc A (Paris):  28.0311
Score for Doc B (Berlin): 19.4445


# 2) Loss Function

ColBERT uses a **pairwise softmax cross-entropy loss**,

### In-batch negatives

For training efficiency, we use **in-batch negatives**: the positive documents of
other queries in the batch serve as negatives. If batch size = B, each query gets
B-1 free negatives. This is very effective and widely used.

ColBERTv2 adds hard negatives mined by a first-stage retriever and distils a cross-encoder's
scores (a KL loss over 64 passages per query). That training needs days of GPU time, so below
the loss is checked on a toy batch and the released `colbert-ir/colbertv2.0` weights are
loaded into the from-scratch module.

In [5]:
class ColBERTLoss(nn.Module):
    """
    Pairwise softmax cross-entropy loss for ColBERT training.

    Supports two modes:
    1. In-batch negatives only: positive documents of other queries serve as negatives.
       scores shape: (batch_num, batch_num) where diagonal = positive pairs.
    2. In-batch negatives + hard negatives: each query has explicit hard negatives.
       scores shape: (batch_num, batch_num * (1 + num_negatives))
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        scores: torch.Tensor,
        positive_indices: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            scores:           (batch_num, num_docs) — similarity scores
            positive_indices: (batch_num,) — index of the positive doc for each query

        Returns:
            loss: scalar — mean cross-entropy loss over the batch
        """

        # Standard cross-entropy: log_softmax over doc dimension, pick the positive
        # (batch_num, num_docs) → scalar
        log_probs = F.log_softmax(scores, dim=-1)

        # Gather the log-probability of the positive document for each query
        # (batch_num, num_docs) → (batch_num, 1) → (batch_num,)
        pos_log_probs = log_probs.gather(
            dim=1, index=positive_indices.unsqueeze(1)
        ).squeeze(1)

        # Negative log-likelihood, averaged over the batch
        loss = -pos_log_probs.mean()

        return loss

In [6]:
# === Test ===
loss_fn = ColBERTLoss()

# Simulate a batch of 3 queries, each with a positive on the diagonal
fake_scores = torch.tensor(
    [
        [5.0, 1.0, 1.0],
        [1.0, 5.0, 1.0],
        [1.0, 1.0, 5.0],
    ],
    dtype=torch.float,
)

# Positive document index = query index (diagonal)
positive_idx = torch.arange(3)

loss_val = loss_fn(fake_scores, positive_idx)
print(f"Loss with high positive scores (should be low): {loss_val.item():.4f}")

# Now with wrong positives scored high — loss should be higher
bad_scores = torch.tensor(
    [
        [1.0, 5.0, 5.0],
        [5.0, 1.0, 5.0],
        [5.0, 5.0, 1.0],
    ],
    dtype=torch.float,
)

loss_val_bad = loss_fn(bad_scores, positive_idx)
print(f"Loss with low positive scores (should be high): {loss_val_bad.item():.4f}")

Loss with high positive scores (should be low): 0.0360
Loss with low positive scores (should be high): 4.7023


# 3) Dataset

**BEIR SciFact**: 5,183 scientific abstracts and 300 test claims, each with the abstracts that
support or refute it as relevance labels. It is small enough to index on a laptop and is part of
the zero-shot BEIR suite on which ColBERTv2 reports nDCG@10 ≈ 0.69 (the checkpoint was trained
on MS MARCO only, so this is out-of-domain).

In [7]:
corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
queries = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_rows = load_dataset("BeIR/scifact-qrels", split="test")

# BEIR convention: index title + abstract
doc_ids = corpus["_id"]
doc_texts = [f"{t} {x}".strip() for t, x in zip(corpus["title"], corpus["text"])]

# {query_id: {doc_id: relevance}} for the 300 test queries
qrels: Dict[str, Dict[str, int]] = {}
for row in qrels_rows:
    qrels.setdefault(str(row["query-id"]), {})[str(row["corpus-id"])] = int(
        row["score"]
    )
query_text = {q["_id"]: q["text"] for q in queries if q["_id"] in qrels}

print(f"{len(doc_texts):,} documents, {len(query_text)} test queries")
example_qid = next(iter(query_text))
print(f"Example query: {query_text[example_qid]!r}")
print(
    f"Relevant doc:  {doc_texts[doc_ids.index(next(iter(qrels[example_qid])))][:150]}..."
)

5,183 documents, 300 test queries
Example query: '0-dimensional biomaterials show inductive properties.'
Relevant doc:  New opportunities: the use of nanotechnologies to manipulate and track stem cells. Nanotechnologies are emerging platforms that could be useful in mea...


# 4) Indexing


ColBERT's efficiency comes from **precomputing document embeddings offline**.
At query time, we only need to encode the query and run MaxSim against stored
embeddings — the expensive BERT forward pass over documents happens only once.

The indexing pipeline:
1. Encode each document through the document encoder
2. Store each token's embedding along with a document ID
3. (Optional) Compress embeddings to reduce storage

### Storage Cost Analysis
For a collection of N documents with average L tokens each:
- **Bi-encoder**: N × d floats (one vector per doc)
- **ColBERT**: N × L × d floats (one vector per token)

With N=8.8M docs (MSMARCO), L≈60 tokens, d=128:
- Bi-encoder: 8.8M × 128 × 4 bytes ≈ 4.5 GB
- ColBERT:    8.8M × 60 × 128 × 4 bytes ≈ 270 GB

This 60x storage increase motivates the residual compression in ColBERTv2.

### Sample Input / Output

```
Input:  ["Paris is the capital of France.", "Berlin is the capital of Germany."]
Output: {
  embeddings: np.array of shape (total_tokens, 128),
  doc_ids:    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],  # which doc each token belongs to
  doc_lengths: [7, 7],  # number of kept tokens per doc
}
```

In [8]:
class ColBERTIndexer:
    """
    Offline indexing pipeline for ColBERT.

    Encodes all documents once and keeps two views of the same token embeddings:
    - a padded (num_docs, max_doc_len, embedding_dim) tensor + mask, for exact MaxSim
    - a flat (total_tokens, embedding_dim) matrix + token→doc map, for candidate generation

    Reference: Khattab & Zaharia (2020), §3.5 (Indexing)
    """

    def __init__(self, model: ColBERT, batch_num: int = 64):
        self.model = model
        self.batch_num = batch_num

    @torch.no_grad()
    def build_index(self, documents: List[str]) -> None:
        """Encode all documents (sorted by length so batches waste little padding)."""
        self.model.eval()
        self.num_docs = len(documents)
        start = time.time()

        # Encode in length order, then put each document back at its original row
        order = sorted(range(len(documents)), key=lambda i: len(documents[i]))
        max_len = self.model.cfg.doc_max_len
        # (num_docs, doc_max_len, embedding_dim) fp16, (num_docs, doc_max_len)
        self.D = torch.zeros(
            self.num_docs,
            max_len,
            self.model.cfg.embedding_dim,
            dtype=torch.float16,
            device=DEVICE,
        )
        self.D_mask = torch.zeros(self.num_docs, max_len, device=DEVICE)
        for batch_start in range(0, len(order), self.batch_num):
            rows = order[batch_start : batch_start + self.batch_num]
            # D: (batch_num, doc_len, embedding_dim), D_mask: (batch_num, doc_len)
            D, D_mask = self.model.encode_document([documents[i] for i in rows])
            rows_t = torch.tensor(rows, device=DEVICE)
            self.D[rows_t, : D.size(1)] = D.half()
            self.D_mask[rows_t, : D.size(1)] = D_mask

        # Flat view of the kept tokens: (total_tokens, embedding_dim) and (total_tokens,)
        keep = self.D_mask > 0
        self.flat = self.D[keep]
        self.flat_doc_ids = torch.arange(self.num_docs, device=DEVICE)[
            :, None
        ].expand_as(keep)[keep]

        print(
            f"Index built in {time.time() - start:.0f}s: {self.num_docs:,} documents, "
            f"{self.flat.shape[0]:,} token embeddings"
        )
        print(
            f"Avg tokens/doc: {self.flat.shape[0] / self.num_docs:.1f}, "
            f"index size: {self.flat.numel() * 2 / 2**20:.0f} MB (fp16)"
        )

In [9]:
indexer = ColBERTIndexer(model)
indexer.build_index(doc_texts)

Index built in 87s: 5,183 documents, 815,632 token embeddings
Avg tokens/doc: 157.4, index size: 199 MB (fp16)


# 5) Retrieval

For each query:
1. Encode the query → (query_max_len, embedding_dim)
2. For each document in the index, gather its token embeddings
3. Compute MaxSim between the query and each document
4. Return the top-k documents by score

### Complexity

For a query with N_q tokens and a collection of D documents averaging L tokens:
- Token similarity computation: O(D × N_q × L × d)
- With d=128, N_q=32, L=60, D=8.8M: ~1.7 trillion multiply-adds per query

In [10]:
class ColBERTRetriever:
    """
    Retrieval engine for ColBERT using a pre-built index.

    Supports:
    1. Brute-force exact retrieval (MaxSim against every document)
    2. Approximate retrieval: candidate generation on single tokens, then exact re-ranking
    """

    def __init__(self, model: ColBERT, indexer: ColBERTIndexer):
        self.model = model
        self.indexer = indexer

    @torch.no_grad()
    def _query(self, query: str) -> torch.Tensor:
        # (query_max_len, embedding_dim) fp16
        return self.model.encode_query([query])[0].half()

    @torch.no_grad()
    def _rerank(
        self, Q: torch.Tensor, doc_rows: torch.Tensor, top_k: int
    ) -> List[Tuple[int, float]]:
        """Exact MaxSim of one query against the given document rows."""
        # (num_candidates, doc_max_len, embedding_dim) and (num_candidates, doc_max_len)
        D, D_mask = self.indexer.D[doc_rows], self.indexer.D_mask[doc_rows]
        # (num_candidates, query_max_len, doc_max_len)
        sim = torch.einsum("qe,dje->dqj", Q, D).float()
        sim = sim.masked_fill(D_mask[:, None, :] == 0, -9999.0)
        # (num_candidates,)
        scores = sim.max(dim=-1).values.sum(dim=-1)
        best = scores.topk(min(top_k, len(scores)))
        return [(int(doc_rows[i]), float(s)) for s, i in zip(best.values, best.indices)]

    def retrieve_bruteforce(
        self, query: str, top_k: int = 10
    ) -> List[Tuple[int, float]]:
        """Score the query against every indexed document; return (doc_row, score)."""
        all_rows = torch.arange(self.indexer.num_docs, device=DEVICE)
        return self._rerank(self._query(query), all_rows, top_k)

    @torch.no_grad()
    def retrieve_approximate(
        self, query: str, top_k: int = 10, nprobe: int = 32
    ) -> List[Tuple[int, float]]:
        """
        Stage 1 — candidate generation: for each query token take its nprobe nearest
        document tokens in the flat index; their documents form the candidate set.
        Stage 2 — exact MaxSim re-ranking of the candidates.

        Production ColBERT runs stage 1 on an IVF/centroid index (FAISS or PLAID) instead of
        this flat search.
        """
        Q = self._query(query)
        # (query_max_len, total_tokens) → (query_max_len, nprobe)
        top_tokens = (Q @ self.indexer.flat.T).topk(nprobe, dim=-1).indices
        candidates = self.indexer.flat_doc_ids[top_tokens.flatten()].unique()
        return self._rerank(Q, candidates, top_k)

In [11]:
retriever = ColBERTRetriever(model, indexer)

for qid in list(query_text)[:3]:
    query = query_text[qid]
    print(f'\nQuery: "{query}"')
    print("-" * 80)

    results = retriever.retrieve_bruteforce(query, top_k=3)
    for rank, (row, score) in enumerate(results, 1):
        relevant = "relevant" if doc_ids[row] in qrels[qid] else "-"
        print(f"  #{rank} (score={score:.2f}, {relevant}): {doc_texts[row][:90]}...")

    # Approximate retrieval for comparison
    approx_rows = {r for r, _ in retriever.retrieve_approximate(query, top_k=3)}
    recall = len(approx_rows & {r for r, _ in results}) / len(results)
    print(f"  Approximate top-3 overlap with exact: {recall:.0%}")


Query: "0-dimensional biomaterials show inductive properties."
--------------------------------------------------------------------------------
  #1 (score=13.31, -): Noninvasive Imaging beyond the Diffraction Limit of 3D Dynamics in Thickly Fluorescent Spe...
  #2 (score=13.11, -): Electrical, optical, and structural properties of indium-tin-oxide thin films for organic ...
  #3 (score=13.11, -): Three-dimensional superresolution colocalization of intracellular protein superstructures ...


  Approximate top-3 overlap with exact: 100%

Query: "1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants."
--------------------------------------------------------------------------------
  #1 (score=19.46, relevant): Rare Variants Create Synthetic Genome-Wide Associations Genome-wide association studies (G...
  #2 (score=19.05, -): Large-Scale Profiling Reveals the Influence of Genetic Variation on Gene Expression in Hum...
  #3 (score=18.92, -): Rare and common variants: twenty arguments Genome-wide association studies have greatly im...


  Approximate top-3 overlap with exact: 100%

Query: "1/2000 in UK have abnormal PrP positivity."
--------------------------------------------------------------------------------
  #1 (score=15.13, -): Prion-induced amyloid heart disease with high blood infectivity in transgenic mice. We inv...
  #2 (score=14.44, -): Detection of prions in blood Prion diseases are caused by an unconventional infectious age...
  #3 (score=14.43, -): Adult T-cell leukemia/lymphoma in London: clinical experience of 21 cases. Adult T-cell le...
  Approximate top-3 overlap with exact: 100%


# 6) Residual Compression (ColBERTv2)

ColBERT's per-token storage is ~60× larger than a bi-encoder.
ColBERTv2 (Santhanam et al., 2022) introduces **residual compression** to
dramatically reduce this cost while preserving retrieval quality.

The key idea:
1. **Cluster** all token embeddings using k-means into C centroids
2. For each embedding, store only its **centroid ID** (log2(C) bits) and a
   **quantized residual** (the difference from its centroid)
3. At retrieval time, reconstruct: embedding ≈ centroid + dequantized_residual

### Compression Ratio
Original: 128 dims × 32 bits = 512 bytes per token
Compressed (2-bit residuals, 65536 centroids):
  - Centroid ID: 16 bits = 2 bytes
  - Residual: 128 dims × 2 bits = 32 bytes
  - Total: 34 bytes per token → **15× compression**

### Sample Input / Output

```
Input embedding:  [0.23, -0.11, 0.45, ...] (128 floats = 512 bytes)
Nearest centroid:  #4217 → [0.20, -0.09, 0.42, ...]
Residual:          [0.03, -0.02, 0.03, ...]
Quantized residual: [01, 00, 01, ...] (2 bits each = 32 bytes)

Reconstructed:     [0.20+0.03, -0.09-0.02, 0.42+0.03, ...]
                 ≈ [0.23, -0.11, 0.45, ...]  (close to original)
```

In [12]:
class ResidualCompressor:
    """
    ColBERTv2-style residual compression for token embeddings.

    1. k-means on (a sample of) token embeddings → centroids
    2. residual = embedding - nearest centroid
    3. quantize each residual dimension to n_bits using quantile bucket boundaries
    Stored per token: one centroid id (uint16) + embedding_dim × n_bits bits.

    Reconstruction:
      embedding_approx = centroid[centroid_id] + bucket_center[code]

    Reference: Santhanam et al., ColBERTv2, §3.2
    """

    def __init__(
        self, n_centroids: int = 4096, n_bits: int = 2, kmeans_iters: int = 10
    ):
        self.n_centroids = n_centroids
        self.n_bits = n_bits
        # Number of quantization levels per dimension
        self.n_levels = 2**n_bits
        self.kmeans_iters = kmeans_iters

    @staticmethod
    def _nearest(
        x: torch.Tensor, centroids: torch.Tensor, chunk: int = 65536
    ) -> torch.Tensor:
        """Nearest centroid per row. Embeddings and centroids are ~unit norm, so the
        largest dot product is the smallest distance. x: (n, d) → (n,)"""
        return torch.cat(
            [
                (x[i : i + chunk] @ centroids.T).argmax(-1)
                for i in range(0, len(x), chunk)
            ]
        )

    def fit(self, embeddings: torch.Tensor, sample_size: int = 65536) -> None:
        """Learn centroids and residual buckets from a random sample of embeddings."""
        start = time.time()
        g = torch.Generator().manual_seed(0)
        sample = embeddings[
            torch.randperm(len(embeddings), generator=g)[:sample_size].to(
                embeddings.device
            )
        ].float()

        # k-means with spherical updates (centroids renormalized, as embeddings are unit norm)
        # (n_centroids, embedding_dim)
        centroids = sample[
            torch.randperm(len(sample), generator=g)[: self.n_centroids].to(
                sample.device
            )
        ].clone()
        for _ in range(self.kmeans_iters):
            assign = self._nearest(sample, centroids)
            sums = torch.zeros_like(centroids).index_add_(0, assign, sample)
            counts = torch.bincount(assign, minlength=self.n_centroids)
            # empty clusters keep their previous centroid
            centroids = torch.where(
                counts[:, None] > 0, F.normalize(sums, dim=-1), centroids
            )
        self.centroids = centroids

        # Quantile bucket boundaries of the residual values (shared by all dimensions)
        residuals = sample - centroids[self._nearest(sample, centroids)]
        flat = residuals.flatten()
        flat = flat[torch.randperm(len(flat), generator=g)[:1_000_000].to(flat.device)]
        # (n_levels + 1,) boundaries, (n_levels,) reconstruction values
        self.bucket_boundaries = torch.quantile(
            flat, torch.linspace(0, 1, self.n_levels + 1, device=flat.device)
        )
        bucket = torch.bucketize(flat, self.bucket_boundaries[1:-1])
        self.bucket_centers = torch.stack(
            [flat[bucket == i].mean() for i in range(self.n_levels)]
        )
        print(
            f"Fitted {self.n_centroids} centroids, {self.n_levels} residual levels "
            f"on {len(sample):,} embeddings in {time.time() - start:.0f}s"
        )

    def compress(self, embeddings: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """(n, d) → centroid ids (n,) and residual codes (n, d) in [0, n_levels)."""
        x = embeddings.float()
        centroid_ids = self._nearest(x, self.centroids)
        codes = torch.bucketize(
            x - self.centroids[centroid_ids], self.bucket_boundaries[1:-1]
        )
        return centroid_ids, codes.to(torch.uint8)

    def decompress(
        self, centroid_ids: torch.Tensor, codes: torch.Tensor
    ) -> torch.Tensor:
        """(n,), (n, d) → reconstructed (n, d) embeddings."""
        return self.centroids[centroid_ids] + self.bucket_centers[codes.long()]

    def stats(
        self, original: torch.Tensor, centroid_ids: torch.Tensor, codes: torch.Tensor
    ) -> Dict[str, float]:
        """Storage with codes bit-packed at n_bits each, plus reconstruction quality."""
        recon = self.decompress(centroid_ids, codes)
        num_tokens, dim = original.shape
        original_bytes = num_tokens * dim * 2
        compressed_bytes = num_tokens * (2 + dim * self.n_bits / 8)
        return {
            "original_mb": original_bytes / 2**20,
            "compressed_mb": compressed_bytes / 2**20,
            "ratio": original_bytes / compressed_bytes,
            "cosine": F.cosine_similarity(original.float(), recon, dim=-1)
            .mean()
            .item(),
        }

In [13]:
# ColBERTv2 picks ~2^floor(log2(16 * sqrt(num_tokens))) centroids; ~4K for this index
compressor = ResidualCompressor(n_centroids=4096, n_bits=2)
compressor.fit(indexer.flat)
centroid_ids, codes = compressor.compress(indexer.flat)

stats = compressor.stats(indexer.flat, centroid_ids, codes)
print(f"fp16 index:          {stats['original_mb']:.0f} MB")
print(
    f"2-bit residual index: {stats['compressed_mb']:.0f} MB  (+ {compressor.centroids.numel() * 4 / 2**20:.0f} MB of centroids)"
)
print(
    f"Compression ratio:    {stats['ratio']:.1f}x versus fp16 ({2 * stats['ratio']:.1f}x versus fp32)"
)
print(f"Mean cosine(original, reconstructed): {stats['cosine']:.4f}")

Fitted 4096 centroids, 4 residual levels on 65,536 embeddings in 1s


fp16 index:          199 MB
2-bit residual index: 26 MB  (+ 2 MB of centroids)
Compression ratio:    7.5x versus fp16 (15.1x versus fp32)
Mean cosine(original, reconstructed): 0.9643


In [14]:
# Swap the reconstructed embeddings into a copy of the index and compare rankings
exact_D = indexer.D
keep = indexer.D_mask > 0
compressed_D = exact_D.clone()
compressed_D[keep] = compressor.decompress(centroid_ids, codes).half()

eval_qids = list(query_text)
overlap = []
for qid in eval_qids:
    indexer.D = exact_D
    exact = {r for r, _ in retriever.retrieve_bruteforce(query_text[qid], top_k=10)}
    indexer.D = compressed_D
    compressed = {
        r for r, _ in retriever.retrieve_bruteforce(query_text[qid], top_k=10)
    }
    overlap.append(len(exact & compressed) / 10)
indexer.D = exact_D
print(
    f"Top-10 overlap, compressed vs exact index, over {len(eval_qids)} queries: {np.mean(overlap):.1%}"
)

Top-10 overlap, compressed vs exact index, over 300 queries: 87.5%


# 7) Evaluation Metrics

We implement the standard evaluation metrics used in the ColBERT papers:

- **MRR@k** (Mean Reciprocal Rank): Average of 1/rank for the first relevant
  document. Standard metric for MSMARCO.
- **Recall@k**: Fraction of queries for which at least one relevant document
  appears in the top-k.
- **nDCG@k** (Normalized Discounted Cumulative Gain): Measures ranking quality
  with graded relevance.

### Sample Input / Output

```
rankings = {"q1": [3, 7, 1], "q2": [1, 4, 2]}
qrels    = {"q1": {1: 1, 3: 1}, "q2": {1: 1}}

MRR@3:    (1/1 + 1/1) / 2 = 1.0
Recall@3: (2/2 + 1/1) / 2 = 1.0  (both queries have all relevant docs in top-3)
```

In [15]:
def compute_mrr(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Mean Reciprocal Rank @ k.

    For each query, find the rank of the first relevant document in the
    top-k results, compute 1/rank, and average across queries.

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]} ordered by score desc
        qrels:    {query_id: {doc_id: relevance_score}} ground truth
        k:        Cutoff depth

    Returns:
        MRR@k score in [0, 1]
    """
    mrr_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        relevant = qrels[qid]
        num_queries += 1

        for rank, doc_id in enumerate(ranked_docs[:k], 1):
            if doc_id in relevant and relevant[doc_id] > 0:
                mrr_sum += 1.0 / rank
                break

    return mrr_sum / max(num_queries, 1)


def compute_recall(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Recall @ k.

    For each query, what fraction of all relevant documents appear in the top-k?

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]} ordered by score desc
        qrels:    {query_id: {doc_id: relevance_score}} ground truth
        k:        Cutoff depth

    Returns:
        Average Recall@k in [0, 1]
    """
    recall_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        relevant = {did for did, rel in qrels[qid].items() if rel > 0}
        if len(relevant) == 0:
            continue

        num_queries += 1
        retrieved_relevant = len(set(ranked_docs[:k]) & relevant)
        recall_sum += retrieved_relevant / len(relevant)

    return recall_sum / max(num_queries, 1)


def compute_ndcg(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Normalized Discounted Cumulative Gain @ k.

    NDCG accounts for graded relevance and rewards placing highly relevant
    documents at higher ranks.

    DCG@k = Σ_{i=1}^{k} (2^{rel_i} - 1) / log2(i + 1)
    NDCG@k = DCG@k / IDCG@k  (where IDCG is the ideal DCG)

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]}
        qrels:    {query_id: {doc_id: relevance_score}}
        k:        Cutoff depth

    Returns:
        Average NDCG@k in [0, 1]
    """

    def dcg(relevances: List[float], k: int) -> float:
        dcg_val = 0.0
        for i, rel in enumerate(relevances[:k]):
            dcg_val += (2**rel - 1) / math.log2(i + 2)
        return dcg_val

    ndcg_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        num_queries += 1

        # Get relevance of each retrieved document
        rels = [qrels[qid].get(doc_id, 0) for doc_id in ranked_docs[:k]]

        # Ideal ranking: sort all relevances descending
        ideal_rels = sorted(qrels[qid].values(), reverse=True)

        dcg_val = dcg(rels, k)
        idcg_val = dcg(ideal_rels, k)

        if idcg_val > 0:
            ndcg_sum += dcg_val / idcg_val

    return ndcg_sum / max(num_queries, 1)

In [16]:
def run_eval(retrieve, qids) -> Dict[str, float]:
    """Retrieve top-10 for every query and compute the standard metrics."""
    rankings = {
        qid: [doc_ids[row] for row, _ in retrieve(query_text[qid], top_k=10)]
        for qid in qids
    }
    return {
        "MRR@10": compute_mrr(rankings, qrels, k=10),
        "Recall@10": compute_recall(rankings, qrels, k=10),
        "nDCG@10": compute_ndcg(rankings, qrels, k=10),
    }


start = time.time()
results = {
    "exact MaxSim": run_eval(retriever.retrieve_bruteforce, eval_qids),
    "candidate gen + rerank": run_eval(retriever.retrieve_approximate, eval_qids),
}
indexer.D = compressed_D
results["2-bit compressed index"] = run_eval(retriever.retrieve_bruteforce, eval_qids)
indexer.D = exact_D

print(f"SciFact test, {len(eval_qids)} queries ({time.time() - start:.0f}s)")
print(f"{'':<26}{'MRR@10':>8}{'Recall@10':>11}{'nDCG@10':>9}")
for name, m in results.items():
    print(f"{name:<26}{m['MRR@10']:>8.3f}{m['Recall@10']:>11.3f}{m['nDCG@10']:>9.3f}")
print("Published ColBERTv2 nDCG@10 on SciFact: ~0.69")

SciFact test, 300 queries (61s)
                            MRR@10  Recall@10  nDCG@10
exact MaxSim                 0.617      0.757    0.646
candidate gen + rerank       0.617      0.757    0.646
2-bit compressed index       0.606      0.749    0.636
Published ColBERTv2 nDCG@10 on SciFact: ~0.69


# Summary

ColBERT occupies a unique point in the retrieval quality vs. efficiency trade-off space:

- **vs. Cross-encoders**: ~100-1000× faster at retrieval (documents pre-encoded)
- **vs. Bi-encoders**: Significantly better quality (token-level matching vs. single vector)
- **vs. BM25**: Better semantic understanding while remaining fast enough for production